In [ ]:
import pandas as pd
import requests
import math
from itertools import permutations

# List of cities
cities_name = ['Tirupati','Visakhapatnam','Vijayawada','Itanagar','Dhubri','Guwahati','Patna','Raipur','Bilaspur','Raigarh','Mopa','Dabolim','Ahmedabad','Surat','Vadodara','Hisar','Shimla','Kullu-Manali','Jamshedpur','Ranchi','Bengaluru','Mysuru','Thiruvananthapuram','Gwalior','Indore','Bhopal','Mumbai','Navi Mumbai','Nashik','Pune','Imphal','Shillong','Puri','Bhubaneswar','Ludhiana','Amritsar','Jaipur','Udaipur','Kota','Gangtok','Coimbatore','Chennai','Hyderabad','Khowai','Ayodhya','Varanasi','Prayagraj','Dehradun','Kolkata','Delhi NCR','Chandigarh','Puducherry','Diu','Srinagar']

# Generate all possible combinations of departure and arrival cities (excluding same city pairs)
city_combinations = list(permutations(cities_name, 2))

# Create DataFrame
df = pd.DataFrame(city_combinations, columns=['Departure', 'Arrival'])

# Cache for city coordinates to avoid redundant API calls
city_coordinates_cache = {}

# Function to fetch city coordinates
def get_city_coordinates(city_name):
    if city_name in city_coordinates_cache:
        return city_coordinates_cache[city_name]
    
    username = 'tisha_agrawal_08'  # Replace with your GeoNames username
    url = f'http://api.geonames.org/searchJSON?q={city_name}&maxRows=1&username={username}'
    
    try:
        response = requests.get(url, timeout=5)
        response.raise_for_status()
        data = response.json()
        
        if 'totalResultsCount' in data and data['totalResultsCount'] > 0:
            city = data['geonames'][0]
            lat, lon = float(city['lat']), float(city['lng'])
            city_coordinates_cache[city_name] = (lat, lon)
            return lat, lon
    except requests.RequestException as e:
        print(f"Error fetching coordinates for {city_name}: {e}")
    
    return None, None

# Function to calculate distance using the Haversine formula
def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # Radius of Earth in km
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = math.sin(dlat / 2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2)**2
    c = 2 * math.asin(math.sqrt(a))
    return R * c  # Distance in km

# Function to calculate flight duration
def calculate_flight_duration(distance, speed=900):
    if distance is None:
        return None
    duration = distance / speed
    hours = int(duration)
    minutes = int((duration - hours) * 60)
    return f"{hours}h {minutes}m"

# Function to process each row
def process_row(row):
    coord1 = get_city_coordinates(row['Departure'])
    coord2 = get_city_coordinates(row['Arrival'])
    
    if coord1 and coord2 and None not in coord1 and None not in coord2:
        distance = haversine(coord1[0], coord1[1], coord2[0], coord2[1])
        duration = calculate_flight_duration(distance)
    else:
        distance, duration = None, None
    
    return pd.Series([distance, duration])

# Apply function to DataFrame
df[['Distance (km)', 'Flight Duration']] = df.apply(process_row, axis=1)

# Replace None with NaN
df.fillna(value=pd.NA, inplace=True)

# Print total combinations
print(f"Total possible routes: {len(df)}")

# Display first 10 rows
print(df.head(10))
